# Data Pull

Pulls all court cases mentioning retired athletes 退役运动员 from China Judgements Online (https://wenshu.court.gov.cn) to a DataFrame. Then exports DataFrame as CSV file "raw_cases.csv" to user's working directory (file already available under the "date" directory in repo).

## Imports

In [1]:
# for scrape via CSS attributes
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "selenium", "pandas", "webdriver-manager", "ipywidgets"])
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

import pandas as pd
import time, re
import matplotlib.pyplot as plt
from urllib.parse import urljoin
from IPython.display import display
import os # for exporting to csv


## print mult things
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

done


## Config Scraper Params
for keywords, query url (COMBINED_QUERY) etc.

In [2]:
KEYWORDS     = ["退役运动员"]  # all ANDed in one search
MAX_PAGES    = None        # set to None to scrape all pages. modify for testing.
LOGIN_WAIT   = 40       # seconds to wait for manual login

BASE_URL      = "https://wenshu.court.gov.cn"
SEARCH_URL    = f"{BASE_URL}/website/wenshu/181029CR4M5A62CH/index.html"
COMBINED_QUERY = " ".join(KEYWORDS)

print(f"Search query : '{COMBINED_QUERY}'")
print(f"Max pages    : {MAX_PAGES}")
print(f"Output file  : {OUTPUT_FILE}")

Search query : '退役运动员'
Max pages    : None
Output file  : cases.csv


In [4]:
# scraper setup

def get_driver(headless=False):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def wait_for(driver, selector, timeout=15):
    return WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, selector))
    )

## Manual Web Log In

Username: 13671006853
Password: Wenshu.Court.Gov0!

In [5]:
driver = get_driver(headless=False)
driver.get(SEARCH_URL)

## Scrape All Pages & Save to df

In [6]:
# enter keywords on page
def search_keywords():
    search_box = driver.find_element(By.CSS_SELECTOR, "input.searchKey")
    driver.execute_script("arguments[0].scrollIntoView(true);", search_box)
    
    time.sleep(0.5)
    search_box.click()
    search_box.clear()
    search_box.send_keys(COMBINED_QUERY)
    print(f"Typed: '{COMBINED_QUERY}'")
    
    search_btn = driver.find_element(By.CSS_SELECTOR, "div.search-rightBtn")
    search_btn.click()
    time.sleep(3)
    first_query_page = driver.current_url
    
    print(f"Current Query Page Timestamp: {first_query_page}")


search_keywords()

Typed: '退役运动员'
Current Query Page: https://wenshu.court.gov.cn/website/wenshu/181217BMTKHNT2W0/index.html?pageId=3051ff968e6b649914e8140d809d8b4c&s21=%E9%80%80%E5%BD%B9%E8%BF%90%E5%8A%A8%E5%91%98


In [7]:
# Paginate through all results & save to df

all_info = []
page = 1

while True:
    if MAX_PAGES and page > MAX_PAGES:
        print(f"Reached page limit ({MAX_PAGES}).")
        break

    print(f"\n <<<<<<<<<< Looking at Page {page} >>>>>>>>>>>>>> ", end=" ")
    
    try:
        wait_for(driver, "div.LM_list")
    except TimeoutException:
        print("No results or timeout.")
    
    cases = driver.find_elements(By.CSS_SELECTOR, "div.LM_list")
    print(f"found {len(cases)} cases")
    
    for case in cases:
        try:
            a = case.find_element(By.CSS_SELECTOR, "a.caseName")
            all_info.append({
                "case_name"  : a.text.strip(),
                "case_type"  : case.find_element(By.CSS_SELECTOR, "div.labelTwo").text.strip(),
                "court"      : case.find_element(By.CSS_SELECTOR, "span.slfyName").text.strip(),
                "date"       : case.find_element(By.CSS_SELECTOR, "span.cprq").text.strip(),
                "case_href"  : urljoin(driver.current_url, a.get_attribute("href")),
                "full_text"  : None
            })
        except NoSuchElementException:
            continue 

    # advance page
    # via: <a class=" pageButton">下一页</a>
    try:
        if driver.find_elements(By.CSS_SELECTOR, "a.disabled pageButton"): # stop if the button is disabled (last page)
            print("Last page reached.")
            break
            
        next_btn = next(
            el for el in driver.find_elements(By.CSS_SELECTOR, "a.pageButton") if el.text.strip() == "下一页" # spaces are ignored in 'class'
        ) # next() picks the first match from the filter

        next_btn.click()
        time.sleep(2)
        page += 1
    except:
        print("reached last page.")
        break

# all_info (list) obtained.
df = pd.DataFrame(all_info)


 <<<<<<<<<< Looking at Page 1 >>>>>>>>>>>>>>  

<selenium.webdriver.remote.webelement.WebElement (session="116e1e542f6a58803868fd8c6a34c795", element="f.2B6E231BE8FACCA00CE2AB283CC8E8AB.d.003FDA0B068722E552A292054587A9B7.e.6419")>

found 5 cases

 <<<<<<<<<< Looking at Page 2 >>>>>>>>>>>>>>  

<selenium.webdriver.remote.webelement.WebElement (session="116e1e542f6a58803868fd8c6a34c795", element="f.2B6E231BE8FACCA00CE2AB283CC8E8AB.d.003FDA0B068722E552A292054587A9B7.e.7314")>

found 5 cases

 <<<<<<<<<< Looking at Page 3 >>>>>>>>>>>>>>  

...

found 5 cases

 <<<<<<<<<< Looking at Page 16 >>>>>>>>>>>>>>  

<selenium.webdriver.remote.webelement.WebElement (session="116e1e542f6a58803868fd8c6a34c795", element="f.2B6E231BE8FACCA00CE2AB283CC8E8AB.d.003FDA0B068722E552A292054587A9B7.e.14923")>

found 5 cases

 <<<<<<<<<< Looking at Page 17 >>>>>>>>>>>>>>  

<selenium.webdriver.remote.webelement.WebElement (session="116e1e542f6a58803868fd8c6a34c795", element="f.2B6E231BE8FACCA00CE2AB283CC8E8AB.d.003FDA0B068722E552A292054587A9B7.e.15455")>

found 1 cases
reached last page.


In [9]:
# check results
df = df.drop_duplicates() # no duplicates.
df.shape
df

(81, 6)

,case_name,case_type,court,date,case_href,full_text
0,北京丰冠体育有限公司等与中国滑雪协会等合同纠纷二审民事判决书,民事二审,北京市高级人民法院,2021-12-09,https://wenshu.court.gov.cn/website/wenshu/181...,None
1,刘慷、吉林省射击射箭运动管理中心人事争议民事再审民事裁定书,民事再审,吉林省高级人民法院,2021-11-05,https://wenshu.court.gov.cn/website/wenshu/181...,None
2,刘小宝、赵效男等买卖合同纠纷民事申请再审审查民事裁定书,民事再审,山东省高级人民法院,2021-07-05,https://wenshu.court.gov.cn/website/wenshu/181...,None
3,马晓三、云南省人民检察院劳动争议再审民事判决书,民事再审,云南省高级人民法院,2020-06-09,https://wenshu.court.gov.cn/website/wenshu/181...,None
4,李素萍与北京市木樨园体育运动技术学校劳动争议再审审查与审判监督民事裁定书,民事审判监督,北京市高级人民法院,2020-06-05,https://wenshu.court.gov.cn/website/wenshu/181...,None
...,...,...,...,...,...,...
76,詹永安与北京市体育局其他一审行政裁定书,行政一审,北京市丰台区人民法院,2015-08-04,https://wenshu.court.gov.cn/website/wenshu/181...,None
77,原告纪程诉通化体育运动学校劳动争议一案一审民事判决书,民事一审,吉林省通化市东昌区人民法院,2015-06-22,https://wenshu.court.gov.cn/website/wenshu/181...,None
78,王钰与被告四川省运动技术学院劳动争议纠纷一案一审民事判决书,民事一审,成都市武侯区人民法院,2015-03-05,https://wenshu.court.gov.cn/website/wenshu/181...,None
79,胡容儿与张建雄、吴佳丽等道路交通事故人身损害赔偿纠纷一审民事判决书,民事一审,杭州市上城区人民法院,2014-04-16,https://wenshu.court.gov.cn/website/wenshu/181...,None


## Export to CSV

In [ ]:
df.to_csv("raw_cases.csv", index = False)
print(f"output CSV file saved to {os.getcwd()}")